# Ders 5: Aktarımlı Öğrenme ve Parametre-Verimli İnce Ayar

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: Ders 2 (parametreler nerede duruyor), Ders 4 (ön eğitimli temsiller).

Büyük bir modelin tam ince ayarı, modelin birkaç katı büyüklüğünde bir optimizasyon durumu saklamayı
ve görev başına ağırlıkların tam bir kopyasını dağıtmayı gerektirir. Parametre-verimli yöntemler —
**LoRA**, adaptörler, önek (prefix) ayarı, BitFit — bunun yerine parametrelerin çok küçük bir kısmını
günceller. Bu defterde bunun neden mümkün olduğunu türetiyor, LoRA'yı ve nicelemeyi sıfırdan
kodluyor ve küçük bir dağıtılabilir modele giden diğer yol olan damıtmayı ele alıyoruz.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. Gerçekte Ne Aktarılıyor?

Öznitelikler ağın altında genel, üstünde göreve özeldir: erken katmanlar her görme görevinin
ihtiyaç duyduğu kenar ve dokuları, geç katmanlar ön eğitim görevinin etiket yapısını öğrenir. Bu,
bilinen pratik kuralı verir — altı dondur, üstü yeniden eğit — ama iki kaydıyla:

- Hedef alan kaynaktan ne kadar **uzaksa** o kadar az katman aktarılır.
- Hedef görevin **verisi** ne kadar çoksa dondurmanın faydası o kadar azalır ve maliyeti artar.

Tam ince ayar ayrıca **felaket boyutunda unutma** riski taşır: ön eğitimli çözümün üzerine yazılır.
Aşağıdaki oyuncak örnekte aynı model önce A görevinde, sonra B görevinde eğitiliyor; ağırlıkları ön
eğitim değerlerine yakın tutan bir cezayla ve o ceza olmadan.


In [ ]:
# İki doğrusal görevin ardışık öğrenilmesi
d, n = 40, 200
rng = np.random.default_rng(1)
wA, wB = rng.normal(size=d), rng.normal(size=d)
XA, XB = rng.normal(size=(n, d)), rng.normal(size=(n, d))
yA, yB = XA@wA, XB@wB

def sgd(X, y, w, steps=400, lr=0.02, anchor=None, lam=0.0):
    w = w.copy()
    for _ in range(steps):
        g = X.T@(X@w - y)/len(X)
        if anchor is not None:
            g = g + lam*(w - anchor)
        w -= lr*g
    return w

w_after_A = sgd(XA, yA, np.zeros(d), steps=2000)
lams = [0.0, 0.05, 0.2, 1.0, 5.0]
errA, errB = [], []
for lam in lams:
    w2 = sgd(XB, yB, w_after_A, steps=2000, anchor=w_after_A, lam=lam)
    errA.append(np.mean((XA@w2 - yA)**2)/np.mean(yA**2))
    errB.append(np.mean((XB@w2 - yB)**2)/np.mean(yB**2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].semilogx(np.array(lams)+1e-3, errA, "o-", lw=2, label="görev A (eski) hatası")
axes[0].semilogx(np.array(lams)+1e-3, errB, "s-", lw=2, label="görev B (yeni) hatası")
axes[0].set_xlabel("ön eğitimli ağırlıklara çekimin gücü (lambda)")
axes[0].set_ylabel("göreli MSE"); axes[0].set_title("Unutma bir kararlılık-esneklik dengesidir")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

layers = ["conv1", "conv2", "conv3", "conv4", "fc"]
near = [0.71, 0.74, 0.76, 0.75, 0.70]
far  = [0.70, 0.68, 0.61, 0.52, 0.44]
x = np.arange(len(layers))
axes[1].plot(x, near, "o-", lw=2, label="hedef alan kaynağa yakın")
axes[1].plot(x, far,  "s-", lw=2, label="hedef alan kaynaktan uzak")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"şuraya kadar dondur\n{l}" for l in layers], fontsize=9)
axes[1].set_ylabel("hedef görev doğruluğu (temsili)")
axes[1].set_title("Öznitelikler ağın ne kadar yukarısına dek yararlı kalıyor")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 2. LoRA: Güncelleme Düşük Ranklıdır

LoRA'nın ardındaki deneysel gözlem şudur: bir ince ayarın ağırlık matrisinde yaptığı *değişimin*
içsel rankı, matrisin kendi rankından çok daha düşüktür. O hâlde $\Delta W \in \mathbb{R}^{d\times k}$
öğrenmek yerine

$$\Delta W = \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k},\ r \ll \min(d,k)$$

öğrenilir. $A$ rastgele, $B$ ise sıfırla ilklenir; böylece eğitim tam olarak ön eğitimli fonksiyondan
başlar. Parametre sayısı $dk$'den $r(d+k)$'ye düşer — $d=k=4096$, $r=8$ için 16.8M'den 65k'ya, yani
$256\times$ azalma. $\alpha/r$ ölçeklemesi, $r$ değiştiğinde etkin güncelleme büyüklüğünü kabaca
sabit tutar; böylece rank başına öğrenme oranını yeniden ayarlamak gerekmez.

Çıkarım sırasında $W + \Delta W$ tek bir matriste **birleştirilebilir**; dolayısıyla adaptörlerin
aksine LoRA hiç gecikme eklemez. Görev başına yalnızca $A, B$ saklandığından tek bir taban model
yüzlerce göreve hizmet edebilir.


In [ ]:
d, k = 256, 256
U = np.linalg.qr(np.random.randn(d, d))[0]
V = np.linalg.qr(np.random.randn(k, k))[0]
# Gerçekçi bir ince ayar güncellemesi: enerji birkaç yönde yoğun, artı küçük bir kuyruk
spec = np.concatenate([np.linspace(1.0, 0.4, 8), 0.05*np.exp(-np.arange(d-8)/40)])
dW_true = U @ np.diag(spec) @ V.T

def lora_approx(M, r):
    Us, S, Vt = np.linalg.svd(M)
    return (Us[:, :r]*S[:r]) @ Vt[:r], r*(M.shape[0]+M.shape[1])

ranks = [1, 2, 4, 8, 16, 32, 64, 128, 256]
errs, params = [], []
for r in ranks:
    approx, p = lora_approx(dW_true, r)
    errs.append(np.linalg.norm(dW_true-approx)/np.linalg.norm(dW_true))
    params.append(p)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].semilogy(np.linalg.svd(dW_true, compute_uv=False), lw=2)
axes[0].set_xlabel("tekil değer indeksi"); axes[0].set_ylabel("büyüklük (log)")
axes[0].set_title("İnce ayar güncellemesi dW'nin spektrumu"); axes[0].grid(alpha=0.3)

axes[1].semilogx(ranks, errs, "o-", lw=2)
axes[1].set_xlabel("LoRA rankı r"); axes[1].set_ylabel("relative reconstruction error")
axes[1].set_title("Rank 8 güncellemenin çoğunu zaten yakalıyor"); axes[1].grid(alpha=0.3, which="both")

axes[2].loglog(ranks, np.array(params)/(d*k), "o-", lw=2, label="LoRA parametreleri / tam")
axes[2].axhline(1.0, ls="--", c="crimson", label="tam ince ayar")
axes[2].set_xlabel("LoRA rankı r"); axes[2].set_ylabel("eğitilen parametre oranı")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3, which="both"); axes[2].set_title("Parametre maliyeti")

plt.tight_layout(); plt.show()

for r in [1, 4, 8, 32]:
    print(f"r={r:3d}: hata {errs[ranks.index(r)]:.3f}   eğitilebilir: dW'nin {params[ranks.index(r)]/ (d*k):.2%} kadarı")


In [ ]:
# Doğrusal bir görevde uçtan uca eğitilen LoRA, birkaç farklı rank ile
d_in, d_out, n = 64, 64, 400
W0 = np.random.randn(d_out, d_in)/np.sqrt(d_in)              # donmuş ön eğitimli ağırlıklar
dW_task = (np.random.randn(d_out, 4) @ np.random.randn(4, d_in))/np.sqrt(d_in)   # gerçek rank-4 kayma
X = np.random.randn(n, d_in)
Y = X @ (W0 + dW_task).T + 0.01*np.random.randn(n, d_out)

def train_lora(r, alpha=None, steps=3000, lr=0.02):
    alpha = alpha if alpha is not None else r
    A = np.random.randn(r, d_in)*0.01
    B = np.zeros((d_out, r))                                  # B = 0  ->  ön eğitimli modelden başlar
    s = alpha/r
    hist = []
    for _ in range(steps):
        H  = X @ A.T                                          # (n, r)
        P  = X @ W0.T + s*(H @ B.T)
        E  = (P - Y)/n
        gB = s*(E.T @ H)
        gA = s*(B.T @ E.T @ X)
        B -= lr*gB; A -= lr*gA
        hist.append(np.mean((P-Y)**2))
    return np.array(hist), s*(B@A)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for r in [1, 2, 4, 8, 16]:
    hist, dW_hat = train_lora(r)
    axes[0].semilogy(hist, lw=1.8, label=f"r={r}  ({r*(d_in+d_out)} parametre)")
axes[0].set_xlabel("adım"); axes[0].set_ylabel("eğitim MSE (log)")
axes[0].set_title("Rank görevin içsel rankına uymalı (gerçek rank = 4)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which="both")

_, dW_hat = train_lora(4)
im0 = axes[1].imshow(np.hstack([dW_task, dW_hat]), cmap="RdBu_r")
axes[1].axvline(d_in-0.5, c="k", lw=2)
axes[1].set_title("Gerçek güncelleme (sol)  ve  öğrenilen LoRA güncellemesi r=4 (sağ)")
axes[1].set_xticks([]); axes[1].set_yticks([]); plt.colorbar(im0, ax=axes[1])
plt.tight_layout(); plt.show()
print(f"r=4'te geri kazanılan güncellemenin göreli hatası: {np.linalg.norm(dW_task-dW_hat)/np.linalg.norm(dW_task):.3f}")


## 3. PEFT Ailesi

| Yöntem | Eğitilen | Ek çıkarım maliyeti | Not |
|---|---|---|---|
| **Tam ince ayar** | her şey | yok | optimizasyon belleği için $\approx 12\times$ model boyutu gerekir (Adam, fp32) |
| **LoRA** | hedef matris başına $BA$ | **yok** (birleştirilir) | varsayılan; ayarlar rank ve *hangi* matrislerin hedefleneceğidir |
| **Adaptörler** | her blokta küçük darboğaz MLP'ler | küçük ama gerçek | sıralı modüller gecikme ekler |
| **Prefix / P-tuning** | öne eklenen sanal anahtar–değer vektörleri | bağlam uzunluğu tüketir | hiç ağırlık değişmez |
| **BitFit** | yalnızca sapmalar (bias) | yok | şaşırtıcı ölçüde güçlü, parametrelerin ~%0.1'i |
| **(IA)³** | öğrenilen ölçekleme vektörleri | yok | LoRA'dan bile az parametre |

Pratikte belirleyici olan bellek argümanıdır. Adam, eğitilebilir her parametre için iki fp32 moment
artı bir fp32 ana kopya saklar; LoRA ile bunlar modelin %0.1'i için ayrılır ve donmuş %99.9'luk
kısım 4 bitte tutulabilir.


In [ ]:
P = 7e9                                       # 7B parametreli model
def memory_gb(trainable_frac, base_bits=16):
    weights = P*base_bits/8
    train   = P*trainable_frac
    return dict(weights=weights/1e9,
                grads=train*4/1e9,
                adam=train*8/1e9,
                master=train*4/1e9)

configs = {"Tam ince ayar (fp16 ağırlık)": (1.0, 16), "LoRA r=8 (fp16 taban)": (0.001, 16),
           "LoRA r=8 (4-bit taban)": (0.001, 4), "BitFit (4-bit taban)": (0.0001, 4)}
labels, stacks = [], []
for name, (frac, bits) in configs.items():
    m = memory_gb(frac, bits); labels.append(name); stacks.append([m["weights"], m["grads"], m["adam"], m["master"]])
stacks = np.array(stacks)

fig, ax = plt.subplots(figsize=(10, 4.4))
bottom = np.zeros(len(labels))
for i, part in enumerate(["donmuş/taban ağırlıklar", "gradyanlar", "Adam momentleri", "fp32 ana kopya"]):
    ax.bar(labels, stacks[:, i], bottom=bottom, label=part)
    bottom += stacks[:, i]
ax.axhline(80, ls="--", c="crimson", lw=1.5, label="80 GB hızlandırıcı")
ax.set_ylabel("eğitim belleği (GB)"); ax.set_title("7B modelin eğitim belleği")
ax.legend(fontsize=9); plt.xticks(rotation=12, fontsize=9); plt.tight_layout(); plt.show()

for name, tot in zip(labels, stacks.sum(1)):
    print(f"{name:26s} {tot:6.1f} GB")


## 4. Niceleme ve QLoRA

QLoRA taban modeli 4 bitte tutar ve üzerinde 16 bitlik LoRA adaptörleri eğitir. 4 biti yaşanabilir
kılan iki ayrıntı vardır:

**Blok bazlı niceleme.** 64 değerlik bloklar hâlinde, blok başına bir ölçekle nicelenir; böylece tek
bir aykırı değer yalnızca kendi bloğuna zarar verir. Transformer aktivasyonları ve ağırlıkları
meşhur biçimde ağır kuyruklu aykırı kanallar barındırır; global tek bir ölçek bunlar tarafından yok
edilirdi.

**NF4 (normal-float 4).** Ağırlıklar yaklaşık Gauss dağılımlı olduğundan, eşit aralıklı seviyeler
yerine normal dağılımın kuantilleri kullanılır; böylece 16 seviyenin her biri olasılık kütlesinden
eşit pay taşır.


In [ ]:
from scipy.stats import norm

def quantize_blockwise(w, bits=4, block=64, levels=None):
    q = np.empty_like(w)
    lv = levels if levels is not None else np.linspace(-1, 1, 2**bits)
    for i in range(0, len(w), block):
        b = w[i:i+block]
        s = np.abs(b).max() + 1e-12
        idx = np.abs(b[:, None]/s - lv[None, :]).argmin(1)
        q[i:i+block] = lv[idx]*s
    return q

# NF4 seviyeleri: normal dağılımın kuantilleri, [-1, 1] aralığına ölçeklenmiş
p = np.linspace(0.5/16, 1-0.5/16, 16)
nf4 = norm.ppf(p); nf4 = nf4/np.abs(nf4).max()

w = np.random.randn(8192)
w[np.random.choice(8192, 20, replace=False)] *= 12                  # aykırı kanallar

results = {
    "int4, tek global ölçek":   quantize_blockwise(w, 4, block=len(w)),
    "int4, 64'lük bloklar":       quantize_blockwise(w, 4, block=64),
    "NF4, 64'lük bloklar":        quantize_blockwise(w, 4, block=64, levels=nf4),
    "int8, 64'lük bloklar":       quantize_blockwise(w, 8, block=64),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
xs = np.linspace(-3, 3, 400)
axes[0].plot(xs, norm.pdf(xs), lw=2, c="k", label="ağırlık dağılımı")
axes[0].scatter(np.linspace(-1, 1, 16), norm.pdf(np.linspace(-1, 1, 16)), s=40, label="int4 seviyeleri (tekdüze)")
axes[0].scatter(nf4, norm.pdf(nf4), s=40, marker="s", label="NF4 seviyeleri (kuantiller)")
axes[0].set_title("16 seviye nereye yerleştiriliyor"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

names = list(results); errs = [np.sqrt(np.mean((w-q)**2))/np.std(w) for q in results.values()]
axes[1].barh(names, errs, color=["#9ecae1", "#6baed6", "#3182bd", "#08519c"])
axes[1].set_xlabel("göreli RMS niceleme hatası"); axes[1].set_title("Blok bazlı ve NF4, ikisi de önemli")

axes[2].scatter(w[:2000], results["int4, tek global ölçek"][:2000], s=4, alpha=0.4, label="global ölçek")
axes[2].scatter(w[:2000], results["NF4, 64'lük bloklar"][:2000], s=4, alpha=0.4, label="NF4 blok bazlı")
axes[2].plot([-4, 4], [-4, 4], "k--", lw=1)
axes[2].set_xlim(-4, 4); axes[2].set_ylim(-4, 4)
axes[2].set_xlabel("orijinal ağırlık"); axes[2].set_ylabel("nicelemesi çözülmüş ağırlık")
axes[2].set_title("Yeniden kurulum kalitesi"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()
for n_, e in zip(names, errs):
    print(f"{n_:26s} göreli RMSE {e:.4f}")
print(f"\n7B ağırlık için bellek: fp16 {7e9*2/1e9:.1f} GB   ->   4-bit {7e9*0.5/1e9:.1f} GB")


## 5. Bilgi Damıtma

PEFT büyük modeli korur ve eğitimi ucuzlatır. Damıtma tam tersini yapar: *küçük* bir modeli büyüğünü
taklit etmesi için eğiterek çıkarımı ucuzlatır. Öğrenci, öğretmenin $T$ sıcaklığındaki **yumuşak**
dağılımına uyar:

$$\mathcal{L} = (1-\lambda)\,\text{CE}(y, p_s) + \lambda T^2\, \text{KL}\big(p_t^{(T)} \,\|\, p_s^{(T)}\big).$$

$T^2$ çarpanı, yumuşak hedef gradyanlarının $1/T^2$ oranında küçülmesini telafi eder ve iki terimi
karşılaştırılabilir ağırlıkta tutar. Yumuşak hedeflerin işe yaramasının nedeni, hedef dışı
olasılıklardaki *karanlık bilgidir*: tek-sıcak (one-hot) bir etiket "bu bir 7'dir" der; öğretmen ise
"bir 7, biraz 1'e benziyor, 8'e hiç benzemiyor" der — örnek başına çok daha zengin bir eğitim
sinyali.


In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True)); return e/e.sum(axis=axis, keepdims=True)

logits_t = np.array([6.0, 2.5, 0.5, -1.0, -2.0])           # tek bir örnek için öğretmen logitleri
classes  = ["7", "1", "9", "4", "8"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(len(classes))
for T, c in zip([1, 2, 4, 8], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[0].plot(x, softmax(logits_t/T), "o-", lw=2, c=c, label=f"T={T}")
axes[0].bar(x, np.eye(len(classes))[0], alpha=0.25, color="crimson", label="tek-sıcak etiket")
axes[0].set_xticks(x); axes[0].set_xticklabels(classes); axes[0].set_yscale("log")
axes[0].set_ylabel("olasılık (log)"); axes[0].set_title("Sıcaklık karanlık bilgiyi açığa çıkarır")
axes[0].legend(fontsize=8)

# Sentetik bir problemde sert etiketle eğitilen öğrenci ile damıtılmış öğrenci
rng = np.random.default_rng(3)
d, K = 20, 5
W_teacher = rng.normal(size=(K, d))
X_pool = rng.normal(size=(300, d))
Xte    = rng.normal(size=(2000, d))
y_te   = (Xte @ W_teacher.T).argmax(1)

def train_student(X, mode, T=4.0, lam=0.9, steps=1500, lr=0.5):
    tl = X @ W_teacher.T                       # eğitim girdilerinde öğretmen logitleri
    y  = tl.argmax(1)                          # sert etiketler de öğretmenden geliyor
    W  = rng.normal(size=(K, d))*0.01
    for _ in range(steps):
        logits = X @ W.T
        g = softmax(logits) - np.eye(K)[y]
        if mode == "damıt":
            g = (1-lam)*g + lam*T*(softmax(logits/T) - softmax(tl/T))
        W -= lr*(g.T @ X)/len(X)
    return np.mean((Xte @ W.T).argmax(1) == y_te)

sizes = [15, 25, 50, 100, 200, 300]
acc_hard = [train_student(X_pool[:m], "sert")    for m in sizes]
acc_soft = [train_student(X_pool[:m], "damıt") for m in sizes]

axes[1].plot(sizes, acc_hard, "o-", lw=2, label="yalnızca sert etiketler")
axes[1].plot(sizes, acc_soft, "s-", lw=2, label="damıtılmış (T=4)")
axes[1].set_xlabel("eğitim örneği sayısı"); axes[1].set_ylabel("öğrenci test doğruluğu")
axes[1].set_title("Yumuşak hedefler ek veriye bedeldir"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].axis("off")
rows = [("logit eşleme", "T -> sonsuz, logitlerde MSE'ye indirger"),
        ("öznitelik damıtma", "yalnızca çıktıyı değil gizli durumları da eşle"),
        ("öz-damıtma", "öğretmen = aynı modelin daha erken bir kopyası"),
        ("dizi düzeyinde damıtma", "öğretmenin ürettiği çıktılar üzerinde eğit")]
axes[2].set_title("Varyantlar", fontsize=12)
for i, (a, b) in enumerate(rows):
    axes[2].text(0.02, 0.85-0.2*i, f"{a}\n   {b}", fontsize=10.5, transform=axes[2].transAxes)

plt.tight_layout(); plt.show()


## 6. Yöntem Seçimi

| Durum | Makul seçim |
|---|---|
| Küçük hedef veri kümesi, taban model zaten iyi | LoRA (düşük rank) ya da donmuş kodlayıcı + doğrusal sonda |
| Çok görev, tek dağıtım | Ortak taban üzerinde istek başına değiştirilen LoRA adaptörleri |
| Tek hızlandırıcı, büyük taban model | QLoRA (4-bit taban + 16-bit adaptör) |
| Büyük hedef veri kümesi, kaynaktan uzak alan | Tam ince ayar — PEFT'in kapasitesi darboğaz olur |
| Kısıt çıkarım maliyeti ise | Daha küçük bir mimariye damıtma, ardından niceleme |

Dikkat edilmesi gereken başarısızlık, sonuncusunun tersidir: PEFT bedava bir öğle yemeği değildir.
Hedef görev, mevcut yeteneklerin yeniden ağırlıklandırılmasından ziyade gerçekten yeni yetenekler
gerektiriyorsa, birkaç matriste rank-8'lik bir güncelleme bunu ifade edemez ve $\alpha$'yı ne kadar
ayarlarsanız ayarlayın bu değişmez.

## 7. Özet

| Kavram | Açıklama |
|---|---|
| **Öznitelik genelliği** | Erken katmanlar geniş biçimde aktarılır; geç katmanlar göreve özeldir |
| **Felaket boyutunda unutma** | Yeni görev eskisinin üzerine yazar; çapa cezaları ikisini dengeler |
| **LoRA** | $\Delta W = \frac{\alpha}{r}BA$, $B$ sıfırla ilklenir, çıkarımda birleştirilebilir |
| **İçsel rank** | İnce ayar güncellemeleri düşük ranklıdır; $r \ll d$ bu yüzden işe yarar |
| **Adaptör / prefix / BitFit** | Farklı ekleme noktaları; farklı gecikme ve kapasite dengeleri |
| **Optimizasyon belleği** | Adam *eğitilebilir* parametre başına ~12 bayt harcar — PEFT'in ana kazancı |
| **Blok bazlı niceleme** | Blok başına ölçekler, aykırı ağırlıkların zararını sınırlar |
| **NF4** | Niceleme seviyeleri normal dağılımın kuantillerinde |
| **Damıtma** | $T$ sıcaklığında yumuşak hedefler; $T^2$ ölçeklemesi; karanlık bilgiyi taşır |

**Sonraki Defter →** Belirsizlik, Kalibrasyon ve Konformal Tahmin
